# Rollback
Rama **FALSE** del health gate. Devuelve el 100% del tráfico al champion actual (alias `champion`) y elimina el served entity challenger del endpoint. **No toca el alias `champion`.** Idempotente.

In [0]:
%pip install -q databricks-sdk mlflow

In [0]:
import mlflow
from mlflow.tracking import MlflowClient
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import serving
mlflow.set_registry_uri("databricks-uc")

MODEL_NAME = dbutils.widgets.get("model_name")
ENDPOINT_NAME = dbutils.widgets.get("endpoint_name")

client = MlflowClient()
w = WorkspaceClient()

try:
    champion_version = client.get_model_version_by_alias(MODEL_NAME, "champion").version
except Exception as e:
    print(f"Rollback: no hay alias 'champion' para revertir (sin champion no se puede rollback). {e}")
    raise

served = [serving.ServedEntityInput(name="champion", entity_name=MODEL_NAME,
                                   entity_version=str(champion_version),
                                   workload_size="Small", scale_to_zero_enabled=True)]
routes = [serving.Route(served_model_name="champion", traffic_percentage="100")]
w.serving_endpoints.update_config(name=ENDPOINT_NAME, served_entities=served,
                                  traffic_config=serving.TrafficConfig(routes=routes))

print(f"Rollback: endpoint {ENDPOINT_NAME} -> 100% champion v{champion_version}")